In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import sys, subprocess, tempfile, shutil, hashlib, urllib.request
import numpy as np

root = Path('/content/drive/MyDrive/CameraTrap_Phase2')
results = root/'phase2_results'
src = results/'final_sprint_v1/code'
parent = results/'guided_crop_trials/20261120T215901842322Z'
species = results/'species_camera_cv/20261120T222021Z'
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(src/'requirements.txt')],check=True)
out = results/'integrated_species_cv'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
local = out/'runtime'
local.mkdir(parents=True)
revision = '1b708b5ae1bb96704459a39c6f6e363b62fa379c'
paths = ('fast_candidate/candidate.py','fast_candidate/crop_sam.py',
         'mask_refinement/combined.py','mask_refinement/bf16_encoder.py',
         'crop_student/crop_model.py','crop_student/guided.py','crop_student/deployment.py',
         'crop_student/best.pt')
for name in paths:
    with urllib.request.urlopen(f'https://raw.githubusercontent.com/salehdev3/Capstone_5/{revision}/experiments/{name}',timeout=90) as response:
        (local/Path(name).name).write_bytes(response.read())
sys.path[:0] = [str(local),str(src)]
for module in ('candidate','crop_sam','combined','bf16_encoder','crop_model','guided','deployment'):
    sys.modules.pop(module,None)
from common import read, save, sha, context, metrics, aggregate, csv_save
from evaluate import benchmark, retained, frame_score
from legacy_evaluation import event_scores
from runtime import DEMO_IDS
from deployment import GuidedPipeline
from train import samples, Crops, infer
from torch.utils.data import DataLoader
from PIL import Image
import torch

old_policy = parent/'224/policy.json'
old_decision = read(parent/'decision.json')
assert sha(old_policy)==old_decision['policy_sha256']=='13a7e7b6b6ec052868736740680fca1cd249ae6071af2fcdf6580dbe039460c7'
policy = read(old_policy)
selection = read(species/'selection.json')
freeze = read(species/'freeze.json')
head = species/'species_head.npz'
assert sha(head)==selection['head_sha256']==freeze['head_sha256']
assert selection['source_checkpoint']==policy['species_classifier']['sha256']
_,_,rows,classes = context(root)
assert selection['classes']==classes
weight = local/'best.pt'
assert sha(weight)=='18ff2c7c2e9d68146a45a1796f96897c8139edab22d57c18717ed26cd81b8b54'
policy['crop_student'] = dict(path=str(weight),sha256=sha(weight))
policy['species_head'] = dict(path=str(head),sha256=sha(head))
policy['species_temperature'] = selection['temperature']
policy['parent_policy_sha256'] = sha(old_policy)
policy['runtime_revision'] = revision
save(out/'policy.json',policy)
by_id = {r['id']:r for r in rows}
cache = Path(tempfile.mkdtemp(prefix='integrated_cv_'))
(cache/'images').mkdir()
def stage(ids):
    for index,ident in enumerate(sorted(set(ids)),1):
        target=cache/'images'/f'{ident}.jpg'
        if not target.exists():
            shutil.copyfile(root/'images'/target.name,target)
            with Image.open(target) as image:
                assert hashlib.sha256(image.convert('RGB').tobytes()).hexdigest()==by_id[ident]['pixel_sha256']
        if index%100==0:
            print('Verified images:',index,flush=True)

save(out/'freeze.json',dict(policy_sha256=sha(out/'policy.json'),head_sha256=sha(head),
                             historical_test_independent=False))
pipe=GuidedPipeline(root,cache,out/'policy.json')
test=retained(rows,'test')
crops=samples(rows,classes,'test')
stage(DEMO_IDS+[r['id'] for r in test+crops])
speed=benchmark(pipe,rows,cache,out/'cpu')
frames={}
scores=[]
for index,row in enumerate(test,1):
    frame=pipe.process_frame(cache/'images'/f"{row['id']}.jpg",row,with_masks=False)
    frames[row['id']]=frame
    scores.append(frame_score(row,[obj['box'] for obj in frame['objects']]))
    if index%50==0:
        print('Detection evaluation:',index,'/',len(test),flush=True)
det=aggregate(scores)
events,excluded=event_scores(rows,frames)
csv_save(out/'detection_per_image.csv',scores)
csv_save(out/'event_counts.csv',events)
# Use the classifier encoder and the same persisted linear head used by the pipeline.
network=pipe.species.net
network.classifier[2]=torch.nn.Identity()
loader=DataLoader(Crops(crops,cache),batch_size=16,num_workers=0)
features,labels=infer(network,loader,'cpu')
head_data=np.load(head)
logits=features@head_data['weight'].T+head_data['bias']
species_metrics=metrics(logits,labels,selection['temperature'])
expected=read(species/'historical_species_metrics.json')
assert len(labels)==expected['n'] and abs(species_metrics['macro_f1']-expected['macro_f1'])<1e-5
save(out/'species_metrics.json',dict(species_metrics,head_sha256=sha(head),
                                     split='historical test',independent_test=False))
targets=[('animal_recall',det['animal_recall'],.99,'min'),
         ('empty_removal',det['empty_removal'],.20,'min'),
         ('species_macro_f1',species_metrics['macro_f1'],.65,'min'),
         ('species_ece',species_metrics['ece'],.10,'max')]
acceptance=[dict(metric=name,value=float(value),target=target,direction=direction,
                 met=bool(value>=target if direction=='min' else value<=target))
            for name,value,target,direction in targets]
csv_save(out/'acceptance_phase1.csv',acceptance)
save(out/'decision.json',dict(decision='GO' if all(item['met'] for item in acceptance) else 'NO-GO',
                              acceptance=acceptance,independent_test=False,policy_sha256=sha(out/'policy.json'),
                              mask_iou_parent=next(x['value'] for x in old_decision['acceptance'] if x['metric']=='mask_iou_min'),
                              mask_scope='inherited from identical detector and segmentation policy, historical test',
                              cpu_p95_seconds=speed['p95'],cpu_scope=speed['scope'],
                              throughput=speed['throughput'],event_count_mae=float(np.mean([x['absolute_error'] for x in events]))))
print('Phase 1 acceptance:',acceptance)
print('CPU p95 seconds:',speed['p95'])
print('Saved:',out)
